# Memória de Cálculo — Capacidade Portuária (Cargas Não Conteinerizadas)

**Roteiro Metodológico para Análise de Capacidades em Terminais Portuários**  
LabPortos/UFMA — Infra S.A.  

Este notebook implementa os Passos 1 e 2 do roteiro metodológico para cargas não conteinerizadas (granéis sólidos vegetais, granéis sólidos minerais, granéis líquidos e carga geral), gerando:

- **Planilha 1** — Base depurada com tempos operacionais calculados (`BD_Depurada.xlsx`)
- **Planilha 2** — Indicadores operacionais por grupo, após depuração por IQR (`Tabela_Indicadores_Operacionais.xlsx`)
- **Planilha 3** — Capacidades calculadas com BOR observado e BUR por berço e perfil (`Tabela_Capacidades.xlsx`)

Os valores das Planilhas 2 e 3 alimentam diretamente o **Quadro 5** (consolidação sistêmica) e o **gráfico-modelo** do Passo 10.

---
**Parâmetros ajustáveis** — antes de executar, revise a célula de configuração abaixo:  
- `BOR_ADM`: taxa de ocupação admissível por tipo de terminal (Quadro 1 do roteiro)  
- `CLEARANCE_H`: intervalo entre atracações consecutivas em horas (padrão: 3,0 h)  
- `H_CAL`: horas do calendário anual (padrão: 8.760)  
- `H_EF`: horas operacionais efetivas (desconta H_cli, H_mnt, H_nav, H_out — Eq. 1c do roteiro)


In [ ]:
# ============================================================
# 0. BIBLIOTECAS
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas.')

In [ ]:
# ============================================================
# 1. PARÂMETROS DE ENTRADA
# Ajuste estes valores conforme o porto analisado e o Quadro 1
# do Roteiro Metodológico.
# ============================================================

# --- Arquivo de entrada ---
ARQUIVO_ENTRADA = 'BD_Bruta_ANTAQ.xlsx'   # base bruta do Estatístico Aquaviário ANTAQ

# --- Parâmetros da Eq. 1b (Roteiro §3.8) ---
CLEARANCE_H = 3.0        # intervalo entre atracações (h); padrão 3,0–3,5 h (Quadro 6)
H_CAL       = 8760       # horas do calendário anual (H_cal = 365 × 24)

# H_ef por porto/berço (Eq. 1c): H_cal − H_cli − H_mnt − H_nav − H_out
# Substitua pelo valor calculado para o porto em análise.
# Enquanto não disponível, use H_CAL como conservador e documente na memória.
H_EF = 8000   # horas operacionais efetivas (valor de referência para terminal sem restrição severa)

# ─────────────────────────────────────────────────────────────────────────
# BOR_adm  (Quadro 17 do Roteiro Metodológico — UNCTAD/PIANC)
# ─────────────────────────────────────────────────────────────────────────
# NOTA DE DIVERGÊNCIA EM RELAÇÃO AO CÓDIGO DE PARANAGUÁ
# O código original dos Planos Mestres (Paranaguá/Maceió) usou:
#     365 * 24 * 0.80 / Tempo_por_Navio  →  BOR_adm = 0,80 fixo, H_ef = 8.760 h
# para TODOS os perfis de carga e TODOS os berços, sem distinção.
#
# Este notebook generaliza esse comportamento de duas formas:
#   1. H_EF substitui 8.760: o analista informa as horas operacionais reais
#      (descontando H_cli, H_mnt, H_nav, H_out — Eq. 1c do roteiro).
#   2. BOR_adm é diferenciado por PERFIL DE CARGA × NÚMERO DE BERÇOS,
#      conforme os limites UNCTAD do Quadro 17 do roteiro.
#      Isso evita a superestimação de capacidade que ocorre quando
#      se aplica 0,80 a terminais de granel sólido com berço único
#      (cujo limite correto é 0,50) ou a terminais de carga geral (0,45).
#
# O objetivo é a maior generalização possível do roteiro:
# o mesmo código deve produzir resultados corretos para qualquer
# porto brasileiro, independentemente do perfil operacional.
# ─────────────────────────────────────────────────────────────────────────

# Mapa BOR_adm por (Perfil da Carga, N.º de berços) — valores UNCTAD (1985)
# Fonte: Quadro 17 do Roteiro Metodológico. Para valores PIANC, somar ~0,05.
# Chave: perfil ANTAQ exato conforme coluna 'Perfil da Carga' da base.
BOR_ADM_MAPA = {
    # (perfil, n_bercos_max) : BOR_adm UNCTAD
    # Granel Sólido
    ('Granel Sólido', 1): 0.50,
    ('Granel Sólido', 2): 0.65,
    ('Granel Sólido', 3): 0.65,
    ('Granel Sólido', 99): 0.65,   # 4+ berços — mesmo teto UNCTAD
    # Granel Líquido e Gasoso
    ('Granel Líquido e Gasoso', 1): 0.55,
    ('Granel Líquido e Gasoso', 2): 0.55,
    ('Granel Líquido e Gasoso', 99): 0.60,
    # Carga Geral
    ('Carga Geral', 1): 0.45,
    ('Carga Geral', 2): 0.60,
    ('Carga Geral', 99): 0.60,
    # Ro-Ro / Automotivo
    ('Ro-Ro', 1): 0.55,
    ('Ro-Ro', 99): 0.55,
}

# Fallback: usado quando o perfil não está no mapa acima.
# 0,80 reproduce o comportamento de Paranaguá para terminais
# multipropósito com 4+ berços — documente se usar este valor.
BOR_ADM_FALLBACK = 0.80

def get_bor_adm(perfil, n_bercos):
    """Retorna BOR_adm pelo mapa Quadro 17; fallback se perfil não mapeado."""
    if n_bercos <= 1:
        chave = (perfil, 1)
    elif n_bercos <= 3:
        chave = (perfil, 2)
    else:
        chave = (perfil, 99)
    return BOR_ADM_MAPA.get(chave, BOR_ADM_FALLBACK)

print('Parâmetros configurados.')
print(f'  H_ef = {H_EF} h/ano | Clearance = {CLEARANCE_H} h')
print(f'  BOR_adm: diferenciado por perfil × berços (Quadro 17) | fallback = {BOR_ADM_FALLBACK}')
print('  Exemplos:')
for ex in [('Granel Sólido',1),('Granel Sólido',2),('Granel Líquido e Gasoso',1),('Carga Geral',1)]:
    print(f'    {ex[0]}, {ex[1]} berço(s): {get_bor_adm(*ex)}')

In [ ]:
# ============================================================
# 2. IMPORTAÇÃO DA BASE BRUTA (Passo 1 do roteiro)
# Fonte: Estatístico Aquaviário ANTAQ — base de cargas não
# conteinerizadas, com 20 colunas e registro por atracação.
# ============================================================
bd_bruta = pd.read_excel(ARQUIVO_ENTRADA)

# Remove colunas auxiliares vazias que podem vir da exportação ANTAQ
bd_bruta = bd_bruta.loc[:, bd_bruta.columns.notna()]

print(f'Base importada: {len(bd_bruta):,} registros | {bd_bruta.shape[1]} colunas')
print('Colunas:', list(bd_bruta.columns))

In [ ]:
# ============================================================
# 3. CHAVE DE AGRUPAMENTO (Apoio Concatenar)
# Concatenação dos campos de identificação do grupo analítico:
# Ano + Terminal + Berço + Nomenclatura + Perfil + Sentido + Navegação.
# Essa chave agrupa todas as atracações com as mesmas características
# operacionais para o cálculo dos indicadores por grupo (Planilha 2).
# ============================================================
bd_bruta['Ano'] = bd_bruta['Ano'].astype(str)
bd_bruta['Apoio Concatenar'] = (
    bd_bruta['Ano'] +
    bd_bruta['Nome do Terminal'] +
    bd_bruta['Berço'] +
    bd_bruta['Nomenclatura Simplificada'] +
    bd_bruta['Perfil da Carga'] +
    bd_bruta['Sentido'] +
    bd_bruta['Navegação da Atracação']
)

print(f'Grupos distintos identificados: {bd_bruta["Apoio Concatenar"].nunique():,}')

In [ ]:
# ============================================================
# 4. FUNÇÃO: tratamento_base  →  PLANILHA 1
# Passo 1 do roteiro (§2.3–§2.5):
#   a) Converte datas e movimentação para tipos numéricos
#   b) Calcula os cinco tempos operacionais (Inop.Pré, T_op,
#      Inop.Pós, Line-up, Produtividade)
#   c) Descarta registros com T_op ≤ 0 (inválidos)
# NOTA: para cargas não conteinerizadas, replicatas são raras;
# o tratamento de replicatas por IMO+timestamps é aplicado
# na base de contêineres (notebook separado).
# ============================================================
def tratamento_base(df):
    """
    Recebe a base bruta da ANTAQ e retorna a base depurada (Planilha 1)
    com colunas de tempos operacionais calculados.
    
    Colunas adicionadas:
        Inop.Pré   : tempo inoperante pré-operação (h)  — Eq. Passo 1
        T_op       : tempo de operação efetiva (h)
        Inop.Pós   : tempo inoperante pós-operação (h)
        Line-up    : tempo de espera no fundeadouro (h) — indicador de nível de serviço
        Produtiv.  : produtividade por atracação (t/h)
    """
    bd = df.copy()

    # --- Conversão de tipos ---
    col_datas = [
        'Data da Chegada', 'Data da Atracação',
        'Data do Início da Operação', 'Data do Término da Operação',
        'Data da Desatracação'
    ]
    for col in col_datas:
        bd[col] = pd.to_datetime(bd[col], errors='coerce')

    col_mov = 'Total de Movimentação Portuária\nem toneladas (t)'
    bd[col_mov] = pd.to_numeric(bd[col_mov], errors='coerce')

    # --- Cálculo dos tempos operacionais (em horas) ---
    bd['Inop.Pré'] = (
        bd['Data do Início da Operação'] - bd['Data da Atracação']
    ).dt.total_seconds() / 3600

    bd['T_op'] = (
        bd['Data do Término da Operação'] - bd['Data do Início da Operação']
    ).dt.total_seconds() / 3600

    bd['Inop.Pós'] = (
        bd['Data da Desatracação'] - bd['Data do Término da Operação']
    ).dt.total_seconds() / 3600

    bd['Line-up'] = (
        bd['Data da Atracação'] - bd['Data da Chegada']
    ).dt.total_seconds() / 3600

    # --- Produtividade (t/h) — apenas para T_op > 0 ---
    bd['Produtividade (t/h)'] = np.where(
        bd['T_op'] > 0,
        bd[col_mov] / bd['T_op'],
        np.nan
    )

    # --- Converte para numérico (elimina eventuais strings residuais) ---
    for col in ['Inop.Pré', 'T_op', 'Inop.Pós', 'Line-up', 'Produtividade (t/h)']:
        bd[col] = pd.to_numeric(bd[col], errors='coerce')

    # --- Descarte de registros inválidos (T_op ≤ 0) ---
    n_antes = len(bd)
    bd_valida = bd[bd['T_op'] > 0].copy()
    n_descartados = n_antes - len(bd_valida)

    print(f'  tratamento_base: {n_antes:,} registros → {len(bd_valida):,} válidos '
          f'| {n_descartados:,} descartados (T_op ≤ 0)')

    # Exclui Carga Conteinerizada (tratada no notebook de contêineres)
    bd_valida = bd_valida[bd_valida['Perfil da Carga'] != 'Carga Conteinerizada']
    print(f'  Após exclusão de contêineres: {len(bd_valida):,} registros')

    return bd_valida


bd_depurada = tratamento_base(bd_bruta)
bd_depurada.head(3)

In [ ]:
# --- Exporta Planilha 1 ---
bd_depurada.to_excel('Planilha1_Base_Depurada.xlsx', index=False)
print('Planilha 1 exportada: Planilha1_Base_Depurada.xlsx')

In [ ]:
# ============================================================
# 5. FUNÇÃO: tabela_indicadores_operacionais  →  PLANILHA 2
# Passo 1–2 do roteiro (§2.5, §3):
#   a) Agrupa por chave Apoio Concatenar
#   b) Calcula Q1, Q3 e limites IQR (L.Inf, L.Sup) para
#      Inop.Pré, Produtividade e Inop.Pós
#   c) Calcula médias depuradas (dentro dos limites IQR)
#   d) Calcula indicadores de movimentação: Lote médio, T_op
#      recalculado, Ta (berth time), Estadia, Movimentação total
#      e número de atracações
# ============================================================
def tabela_indicadores_operacionais(bd):
    """
    Recebe a base depurada (Planilha 1) e retorna a tabela de
    indicadores operacionais por grupo (Planilha 2).

    Uma linha por combinação (Terminal, Berço, Nomenclatura,
    Perfil, Sentido, Navegação).
    """
    col_mov = 'Total de Movimentação Portuária\nem toneladas (t)'

    # --- Tabela-base: uma linha por grupo ---
    cols_id = [
        'Apoio Concatenar', 'Complexo Portuário', 'Nome do Terminal',
        'Berço', 'Ano', 'Perfil da Carga', 'Nomenclatura Simplificada',
        'Sentido', 'Navegação da Atracação'
    ]
    tab = bd[cols_id].drop_duplicates(subset='Apoio Concatenar').copy()

    # --- IQR: Inop.Pré ---
    q1_pre  = bd.groupby('Apoio Concatenar')['Inop.Pré'].quantile(0.25)
    q3_pre  = bd.groupby('Apoio Concatenar')['Inop.Pré'].quantile(0.75)
    tab['Q1_Inop.Pré']    = tab['Apoio Concatenar'].map(q1_pre)
    tab['Q3_Inop.Pré']    = tab['Apoio Concatenar'].map(q3_pre)
    tab['L.Inf_Inop.Pré'] = tab['Q1_Inop.Pré'] - 1.5 * (tab['Q3_Inop.Pré'] - tab['Q1_Inop.Pré'])
    tab['L.Sup_Inop.Pré'] = tab['Q3_Inop.Pré'] + 1.5 * (tab['Q3_Inop.Pré'] - tab['Q1_Inop.Pré'])

    # --- IQR: Produtividade ---
    q1_prod = bd.groupby('Apoio Concatenar')['Produtividade (t/h)'].quantile(0.25)
    q3_prod = bd.groupby('Apoio Concatenar')['Produtividade (t/h)'].quantile(0.75)
    tab['Q1_Produtiv.']    = tab['Apoio Concatenar'].map(q1_prod)
    tab['Q3_Produtiv.']    = tab['Apoio Concatenar'].map(q3_prod)
    tab['L.Inf_Produtiv.'] = tab['Q1_Produtiv.'] - 1.5 * (tab['Q3_Produtiv.'] - tab['Q1_Produtiv.'])
    tab['L.Sup_Produtiv.'] = tab['Q3_Produtiv.'] + 1.5 * (tab['Q3_Produtiv.'] - tab['Q1_Produtiv.'])

    # --- IQR: Inop.Pós ---
    q1_pos  = bd.groupby('Apoio Concatenar')['Inop.Pós'].quantile(0.25)
    q3_pos  = bd.groupby('Apoio Concatenar')['Inop.Pós'].quantile(0.75)
    tab['Q1_Inop.Pós']    = tab['Apoio Concatenar'].map(q1_pos)
    tab['Q3_Inop.Pós']    = tab['Apoio Concatenar'].map(q3_pos)
    tab['L.Inf_Inop.Pós'] = tab['Q1_Inop.Pós'] - 1.5 * (tab['Q3_Inop.Pós'] - tab['Q1_Inop.Pós'])
    tab['L.Sup_Inop.Pós'] = tab['Q3_Inop.Pós'] + 1.5 * (tab['Q3_Inop.Pós'] - tab['Q1_Inop.Pós'])

    # --- Merge dos limites IQR na base para filtro ---
    bd2 = bd.merge(
        tab[['Apoio Concatenar',
             'L.Inf_Inop.Pré', 'L.Sup_Inop.Pré',
             'L.Inf_Produtiv.', 'L.Sup_Produtiv.',
             'L.Inf_Inop.Pós', 'L.Sup_Inop.Pós']],
        on='Apoio Concatenar', how='left'
    )

    # --- Função de média dentro dos limites ---
    def media_iqr(grupo_df, col, col_inf, col_sup):
        """Média dos valores dentro do intervalo [L.Inf, L.Sup]."""
        mask = (grupo_df[col] >= grupo_df[col_inf]) & (grupo_df[col] <= grupo_df[col_sup])
        return grupo_df.loc[mask, col].mean()

    # Aplica para cada grupo
    media_pre  = {}
    media_prod = {}
    media_pos  = {}
    for chave, grp in bd2.groupby('Apoio Concatenar'):
        media_pre[chave]  = media_iqr(grp, 'Inop.Pré',       'L.Inf_Inop.Pré',  'L.Sup_Inop.Pré')
        media_prod[chave] = media_iqr(grp, 'Produtividade (t/h)', 'L.Inf_Produtiv.', 'L.Sup_Produtiv.')
        media_pos[chave]  = media_iqr(grp, 'Inop.Pós',       'L.Inf_Inop.Pós',  'L.Sup_Inop.Pós')

    tab['Média_Inop.Pré (h)']      = tab['Apoio Concatenar'].map(media_pre)
    tab['Média_Produtiv. (t/h)']   = tab['Apoio Concatenar'].map(media_prod)
    tab['Média_Inop.Pós (h)']      = tab['Apoio Concatenar'].map(media_pos)

    # --- Indicadores de movimentação ---
    agg = bd.groupby('Apoio Concatenar').agg(
        Lote_medio     = (col_mov, 'mean'),
        Movimentacao   = (col_mov, 'sum'),
        Estadia_media  = ('Line-up', 'mean'),
        Atracacoes     = ('Data da Atracação', 'count'),
        N_bercos       = ('Berço', 'nunique')
    ).reset_index()
    agg.columns = [
        'Apoio Concatenar', 'Lote médio (t)', 'Movimentação total (t)',
        'Line-up médio (h)', 'N.º atracações', 'N.º berços distintos'
    ]
    tab = tab.merge(agg, on='Apoio Concatenar', how='left')

    # --- Tempos derivados (Eq. 1b — Roteiro §3.8) ---
    # T_op recalculado = Lote médio / Produtividade média depurada
    tab['T_op recalc. (h)']  = tab['Lote médio (t)'] / tab['Média_Produtiv. (t/h)']
    # Ta = Inop.Pré + T_op + Inop.Pós (berth time)
    tab['Ta — berth time (h)'] = (
        tab['Média_Inop.Pré (h)'] +
        tab['T_op recalc. (h)'] +
        tab['Média_Inop.Pós (h)']
    )
    # Clearance fixo
    tab['Clearance a (h)'] = CLEARANCE_H
    tab['Ta + a (h)']      = tab['Ta — berth time (h)'] + CLEARANCE_H

    n_grupos = len(tab)
    print(f'  tabela_indicadores: {n_grupos:,} grupos (linhas na Planilha 2)')
    return tab


tab_indicadores = tabela_indicadores_operacionais(bd_depurada)
tab_indicadores.head(3)

In [ ]:
# --- Exporta Planilha 2 ---
tab_indicadores.to_excel('Planilha2_Indicadores_Operacionais.xlsx', index=False)
print('Planilha 2 exportada: Planilha2_Indicadores_Operacionais.xlsx')

In [ ]:
# ============================================================
# 6. FUNÇÃO: tabela_capacidades_calc  →  PLANILHA 3
# Passo 2 do roteiro (§3.8–§3.11):
#   a) Calcula capacidade bruta pela Eq. 1b
#   b) Realiza alocação por mix de utilização do berço (§3.10)
#   c) Calcula BOR observado (Eq. 2a) e BUR observado (Eq. 2b)
#   d) Sinaliza grupos acima do BOR_adm (possível gargalo)
# ============================================================
def tabela_capacidades_calc(tab_ind):
    """
    Recebe a tabela de indicadores (Planilha 2) e retorna a tabela
    de capacidades calculadas (Planilha 3).

    Equações aplicadas:
        Eq. 1b  C_cais = (b × BOR_adm × H_ef × Lm) / (Ta + a)
        Eq. 2a  BOR    = (Σ Ta) / (b × H_ef) × 100
        Eq. 2b  BUR    = Movimentação_realizada / C_cais × 100
    """
    cols = [
        'Apoio Concatenar', 'Ano', 'Complexo Portuário', 'Nome do Terminal',
        'Berço', 'Perfil da Carga', 'Nomenclatura Simplificada',
        'Sentido', 'Navegação da Atracação',
        'Lote médio (t)', 'Movimentação total (t)',
        'N.º atracações', 'N.º berços distintos',
        'Média_Inop.Pré (h)', 'T_op recalc. (h)', 'Média_Inop.Pós (h)',
        'Ta — berth time (h)', 'Clearance a (h)', 'Ta + a (h)'
    ]
    tab = tab_ind[cols].copy()

    # --- BOR_adm: diferenciado por perfil × número de berços (Quadro 17) ---
    # Aplica get_bor_adm() definida na célula de parâmetros.
    # DIFERENÇA EM RELAÇÃO AO CÓDIGO DE PARANAGUÁ:
    #   Paranaguá: 0,80 fixo, H_ef = 8.760 h (calendário), todos os perfis.
    #   Este notebook: limite específico por perfil × berços (Quadro 17) e H_EF real.
    tab['BOR_adm'] = tab.apply(
        lambda row: get_bor_adm(
            row['Perfil da Carga'],
            int(row['N.º berços distintos']) if pd.notna(row['N.º berços distintos']) else 1
        ), axis=1
    )
    tab['BOR_adm fonte'] = tab['Perfil da Carga'].apply(
        lambda p: 'Quadro 17 (UNCTAD)' if any(p == k[0] for k in BOR_ADM_MAPA) else f'Fallback {BOR_ADM_FALLBACK}'
    )

    # --- Eq. 1b: Capacidade bruta por grupo ---
    # C_cais = BOR_adm × H_ef × Lm / (Ta + a)
    tab['C_cais bruta (t/ano)'] = (
        tab['BOR_adm'] * H_EF * tab['Lote médio (t)']
    ) / tab['Ta + a (h)']

    # --- Alocação por mix de utilização do berço (§3.10) ---
    # T_total(j) = (Mov(j) / Lm(j)) × (Ta(j) + a)
    tab['T_total_j (h)'] = (
        tab['Movimentação total (t)'] / tab['Lote médio (t)']
    ) * tab['Ta + a (h)']

    # Soma de T_total por berço
    t_bercо = tab.groupby('Berço')['T_total_j (h)'].transform('sum')
    tab['T_total_berço (h)'] = t_bercо

    # Fração f(j) = T_total(j) / T_total_berço
    tab['f(j) — fração mix'] = tab['T_total_j (h)'] / tab['T_total_berço (h)']

    # Capacidade alocada C(j) = C_bruta × f(j)
    c_bruta_bercо = tab.groupby('Berço')['C_cais bruta (t/ano)'].transform('first')
    tab['C(j) alocada (t/ano)'] = c_bruta_bercо * tab['f(j) — fração mix']

    # --- Eq. 2a: BOR observado ---
    # BOR = (N_atracações × Ta) / (N_berços × H_ef)
    tab['BOR observado (%)'] = (
        tab['N.º atracações'] * tab['Ta — berth time (h)']
    ) / (tab['N.º berços distintos'] * H_EF) * 100

    # --- Eq. 2b: BUR observado ---
    # BUR = Movimentação_realizada / C(j)
    tab['BUR observado (%)'] = (
        tab['Movimentação total (t)'] / tab['C(j) alocada (t/ano)']
    ) * 100

    # --- Sinalização: BOR acima do admissível ---
    tab['BOR > BOR_adm?'] = tab['BOR observado (%)'] > (tab['BOR_adm'] * 100)

    print(f'  tabela_capacidades: {len(tab):,} linhas na Planilha 3')
    grupos_saturados = tab['BOR > BOR_adm?'].sum()
    if grupos_saturados > 0:
        print(f'  ATENÇÃO: {grupos_saturados} grupo(s) com BOR observado > BOR_adm — verificar gargalos.')
    return tab


tab_capacidades = tabela_capacidades_calc(tab_indicadores)
tab_capacidades.head(3)

In [ ]:
# --- Exporta Planilha 3 ---
tab_capacidades.to_excel('Planilha3_Capacidades.xlsx', index=False)
print('Planilha 3 exportada: Planilha3_Capacidades.xlsx')

In [ ]:
# ============================================================
# 7. SÍNTESE PARA O QUADRO 5 DO ROTEIRO
# Agrupa a Planilha 3 por Terminal + Perfil de Carga, somando
# capacidades alocadas e movimentação para produzir os valores
# que alimentam o Quadro 5 (consolidação sistêmica).
# ============================================================
quadro5 = (
    tab_capacidades
    .groupby(['Complexo Portuário', 'Nome do Terminal', 'Berço', 'Perfil da Carga', 'Nomenclatura Simplificada'])
    .agg(
        C_cais_ano       = ('C(j) alocada (t/ano)', 'sum'),
        Movimentacao_ano = ('Movimentação total (t)', 'sum'),
        BOR_obs_pct      = ('BOR observado (%)', 'mean'),
        BOR_adm_pct      = ('BOR_adm', 'mean'),
        BUR_obs_pct      = ('BUR observado (%)', 'mean'),
        N_atracacoes     = ('N.º atracações', 'sum'),
    )
    .reset_index()
)
quadro5.columns = [
    'Complexo Portuário', 'Terminal', 'Berço', 'Perfil de Carga', 'Nomenclatura',
    'C_cais (t/ano)', 'Movimentação observada (t/ano)',
    'BOR obs. (%)', 'BOR adm. (%)', 'BUR obs. (%)', 'N.º atracações'
]
quadro5['BOR adm. (%)'] = quadro5['BOR adm. (%)'] * 100

quadro5.to_excel('Quadro5_Consolidacao_Sistemica.xlsx', index=False)
print('Quadro 5 exportado: Quadro5_Consolidacao_Sistemica.xlsx')
quadro5

In [ ]:
# ============================================================
# 8. LOG DE DEPURAÇÃO
# Registra, por grupo, o número de observações totais vs.
# retidas no IQR, para rastreabilidade da memória de cálculo
# (exigência §11 do roteiro).
# ============================================================
bd2 = bd_depurada.merge(
    tab_indicadores[[
        'Apoio Concatenar',
        'L.Inf_Inop.Pré','L.Sup_Inop.Pré',
        'L.Inf_Produtiv.','L.Sup_Produtiv.',
        'L.Inf_Inop.Pós','L.Sup_Inop.Pós'
    ]],
    on='Apoio Concatenar', how='left'
)

def contar_retidos(grp, col, col_inf, col_sup):
    mask = (grp[col] >= grp[col_inf]) & (grp[col] <= grp[col_sup])
    return mask.sum()

log_rows = []
for chave, grp in bd2.groupby('Apoio Concatenar'):
    log_rows.append({
        'Apoio Concatenar': chave,
        'N total': len(grp),
        'N retidos Inop.Pré':  contar_retidos(grp, 'Inop.Pré',  'L.Inf_Inop.Pré',  'L.Sup_Inop.Pré'),
        'N retidos Produtiv.': contar_retidos(grp, 'Produtividade (t/h)', 'L.Inf_Produtiv.', 'L.Sup_Produtiv.'),
        'N retidos Inop.Pós':  contar_retidos(grp, 'Inop.Pós',  'L.Inf_Inop.Pós',  'L.Sup_Inop.Pós'),
    })

log_df = pd.DataFrame(log_rows)
log_df.to_excel('Log_Depuracao_IQR.xlsx', index=False)
print('Log de depuração exportado: Log_Depuracao_IQR.xlsx')
print(f'Total de grupos no log: {len(log_df):,}')

---
## Arquivos gerados

| Arquivo | Corresponde a | Uso no roteiro |
|---|---|---|
| `Planilha1_Base_Depurada.xlsx` | Planilha 1 (§11.2) | Base com tempos calculados; insumo para Planilha 2 |
| `Planilha2_Indicadores_Operacionais.xlsx` | Planilha 2 (§11.2) | Indicadores depurados por grupo; insumo para Planilha 3 |
| `Planilha3_Capacidades.xlsx` | Planilha 3 (§11.2) | C_cais, BOR, BUR por berço e perfil; alimenta Quadro 5 |
| `Quadro5_Consolidacao_Sistemica.xlsx` | Quadro 5 (§9.2) | Template de consolidação sistêmica preenchido |
| `Log_Depuracao_IQR.xlsx` | Log (§11.3) | Rastreabilidade da depuração por IQR |

**Próximos passos:**  
- Preencha C_arm e C_hint no `Quadro5_Consolidacao_Sistemica.xlsx` com os resultados dos Passos 4–7  
- Aplique a Eq. 13 (C_sistema = min(C_cais, C_arm, C_hint)) para identificar o elo restritivo  
- Use o notebook de contêineres para a base `BD_Bruta_ANTAQ_Cont.xlsx`
